# Data Cleaning


## 0. Загружаем данные

Сохраняем неизменяемую копию исходного датасета в `raw_df`. Все эксперименты и преобразования делаем только с `df`.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

raw_df = pd.read_csv('../data/raw/hotel_bookings.csv')
df = raw_df.copy()

print("Raw shape:", raw_df.shape)
df.head()


Raw shape: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## 1. Удаляем признаки с leakage

Наша постановка задачи: **предсказать отмену сразу после создания бронирования**.

Поэтому удаляем признаки, которые в этот момент ещё неизвестны или напрямую сообщают итог бронирования:


In [2]:
leakage_cols = [
    'reservation_status',
    'reservation_status_date',
    'booking_changes',
    'assigned_room_type',
    'days_in_waiting_list'
]

df = df.drop(columns=leakage_cols)
df.shape

(119390, 27)

## 2. Пропуски

Сначала посмотрим количество и долю пропусков.


In [3]:
missing = pd.DataFrame({
    'count': df.isna().sum(),
    'percent': df.isna().mean() * 100
})

missing[missing['count'] > 0].sort_values('percent', ascending=False)

,count,percent
company,112593,94.306893
agent,16340,13.686238
country,488,0.408744
children,4,0.003350


### `children`

В `children` всего несколько пропусков. Пока не удаляем сам признак. Для бинарного признака `has_children` безопасно считаем пропуск как 0 только внутри выражения, но исходный `children` сохраняем.


In [4]:
df['has_children'] = (
    df['children'].fillna(0) + df['babies']
    > 0
).astype(int)

df[['children', 'babies', 'has_children']].head()

,children,babies,has_children
0,0.0,0,0
1,0.0,0,0
2,0.0,0,0
3,0.0,0,0
4,0.0,0,0


### `agent` и `company`

`agent` и `company` — это идентификаторы категорий, а не количественные величины. 

Пропуск здесь может быть информативен: возможно, бронь сделана без агента или без компании. Поэтому создаём отдельные бинарные признаки наличия агента/компании и **не складываем значения ID**.


In [5]:
df['has_agent'] = df['agent'].notna().astype(int)
df['has_company'] = df['company'].notna().astype(int)
df['has_agent_or_company'] = (
    df['agent'].notna() | df['company'].notna()
).astype(int)

df[['agent', 'company', 'has_agent', 'has_company', 'has_agent_or_company']].head(10)

,agent,company,has_agent,has_company,has_agent_or_company
0,NaN,NaN,0,0,0
1,NaN,NaN,0,0,0
2,NaN,NaN,0,0,0
3,304.0,NaN,1,0,1
4,240.0,NaN,1,0,1
5,240.0,NaN,1,0,1
6,NaN,NaN,0,0,0
7,303.0,NaN,1,0,1
8,240.0,NaN,1,0,1
9,15.0,NaN,1,0,1


Сами `agent` и `company` переводим в категориальный тип, потому что это ID категорий.


In [6]:
for col in ["agent", "company"]:
    df[col] = (
        df[col]
        .astype("Int64")
        .astype("string")
    )

df[["agent", "company"]].dtypes

agent      string
company    string
dtype: object

### `country`

В `country` пропуск означает, что страна неизвестна. Пока сохраняем эти строки и заменяем `NaN` на отдельную категорию `Unknown`.


In [7]:
df['country'] = df['country'].fillna('Unknown')

df['country'].value_counts(dropna=False).head(15)

country
PRT    48590
GBR    12129
FRA    10415
ESP     8568
DEU     7287
ITA     3766
IRL     3375
BEL     2342
BRA     2224
NLD     2104
USA     2097
CHE     1730
CN      1279
AUT     1263
SWE     1024
Name: count, dtype: int64

## 3. Категориальные признаки

Важно смотреть `value_counts()` **для каждого признака отдельно**, а не для комбинации всех колонок.


In [8]:
cat_cols = df.select_dtypes(
    include=['object', 'string', 'category']
).columns.tolist()

for col in cat_cols:
    print(f"\n--- {col} ---")
    print("nunique:", df[col].nunique(dropna=False))
    print(df[col].value_counts(dropna=False).head(20))


--- hotel ---
nunique: 2
hotel
City Hotel      79330
Resort Hotel    40060
Name: count, dtype: int64

--- arrival_date_month ---
nunique: 12
arrival_date_month
August       13877
July         12661
May          11791
October      11160
April        11089
June         10939
September    10508
March         9794
February      8068
November      6794
December      6780
January       5929
Name: count, dtype: int64

--- meal ---
nunique: 5
meal
BB           92310
HB           14463
SC           10650
Undefined     1169
FB             798
Name: count, dtype: int64

--- country ---
nunique: 178
country
PRT    48590
GBR    12129
FRA    10415
ESP     8568
DEU     7287
ITA     3766
IRL     3375
BEL     2342
BRA     2224
NLD     2104
USA     2097
CHE     1730
CN      1279
AUT     1263
SWE     1024
CHN      999
POL      919
ISR      669
RUS      632
NOR      607
Name: count, dtype: int64

--- market_segment ---
nunique: 8
market_segment
Online TA        56477
Offline TA/TO    24219
Groups        

### Что отдельно проверить

- `Undefined` в `meal` и `distribution_channel` — пока считаем отдельной категорией, а не ошибкой.
- `country` имеет высокую кардинальность, но пока не удаляем его.
- Позже можно сравнить исходный `country` и дополнительный бинарный признак `is_domestic`.


In [9]:
df["country_group"] = np.where(
    df["country"] == "Unknown",
    "Unknown",
    np.where(
        df["country"] == "PRT",
        "Domestic",
        "Foreign"
    )
)

## 4. Числовые признаки

Смотрим диапазоны. Не делаем вывод «всё нормально» только по тому, что тип данных числовой.


In [10]:
numeric_cols = df.select_dtypes(include=['int', 'float']).columns.tolist()

df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
is_canceled,119390.0,0.370416,0.482918,0.00,0.00,0.000,1.0,1.0
lead_time,119390.0,104.011416,106.863097,0.00,18.00,69.000,160.0,737.0
arrival_date_year,119390.0,2016.156554,0.707476,2015.00,2016.00,2016.000,2017.0,2017.0
arrival_date_week_number,119390.0,27.165173,13.605138,1.00,16.00,28.000,38.0,53.0
arrival_date_day_of_month,119390.0,15.798241,8.780829,1.00,8.00,16.000,23.0,31.0
stays_in_weekend_nights,119390.0,0.927599,0.998613,0.00,0.00,1.000,2.0,19.0
stays_in_week_nights,119390.0,2.500302,1.908286,0.00,1.00,2.000,3.0,50.0
adults,119390.0,1.856403,0.579261,0.00,2.00,2.000,2.0,55.0
children,119386.0,0.103890,0.398561,0.00,0.00,0.000,0.0,10.0
babies,119390.0,0.007949,0.097436,0.00,0.00,0.000,0.0,10.0


### Исследуем потенциальные выбросы и аномальные значения

Особенно интересуют `adr`, `adults`, `babies`, `lead_time`, а также длительности проживания.


In [11]:
cols_to_check = [
    'adr',
    'adults',
    'babies',
    'lead_time',
    'stays_in_week_nights',
    'stays_in_weekend_nights'
]

for col in cols_to_check:
    print(f"\n--- {col} ---")
    print(df[col].quantile([0, 0.5, 0.95, 0.99, 0.995, 0.999, 1.0]))


--- adr ---
0.000      -6.38000
0.500      94.57500
0.950     193.50000
0.990     252.00000
0.995     275.00000
0.999     326.20163
1.000    5400.00000
Name: adr, dtype: float64

--- adults ---
0.000     0.0
0.500     2.0
0.950     3.0
0.990     3.0
0.995     3.0
0.999     3.0
1.000    55.0
Name: adults, dtype: float64

--- babies ---
0.000     0.0
0.500     0.0
0.950     0.0
0.990     0.0
0.995     1.0
0.999     1.0
1.000    10.0
Name: babies, dtype: float64

--- lead_time ---
0.000      0.00
0.500     69.00
0.950    320.00
0.990    444.00
0.995    476.11
0.999    605.00
1.000    737.00
Name: lead_time, dtype: float64

--- stays_in_week_nights ---
0.000     0.0
0.500     2.0
0.950     5.0
0.990    10.0
0.995    10.0
0.999    19.0
1.000    50.0
Name: stays_in_week_nights, dtype: float64

--- stays_in_weekend_nights ---
0.000     0.0
0.500     1.0
0.950     2.0
0.990     4.0
0.995     4.0
0.999     6.0
1.000    19.0
Name: stays_in_weekend_nights, dtype: float64


Посмотрим экстремальные значения `adr` отдельно.


In [12]:
df.nsmallest(10, 'adr')[
    ['hotel', 'is_canceled', 'adr', 'adults', 'children', 'babies',
     'stays_in_week_nights', 'stays_in_weekend_nights']
]

,hotel,is_canceled,adr,adults,children,babies,stays_in_week_nights,stays_in_weekend_nights
14969,Resort Hotel,0,-6.38,2,0.0,0,6,4
0,Resort Hotel,0,0.00,2,0.0,0,0,0
1,Resort Hotel,0,0.00,2,0.0,0,0,0
125,Resort Hotel,0,0.00,4,0.0,0,1,0
167,Resort Hotel,0,0.00,2,0.0,0,0,0
168,Resort Hotel,0,0.00,1,0.0,0,0,0
196,Resort Hotel,0,0.00,2,0.0,0,0,0
197,Resort Hotel,0,0.00,2,0.0,0,0,0
421,Resort Hotel,1,0.00,2,0.0,0,2,0
428,Resort Hotel,0,0.00,1,0.0,0,2,0


In [13]:
df.nlargest(10, 'adr')[
    ['hotel', 'is_canceled', 'adr', 'adults', 'children', 'babies',
     'stays_in_week_nights', 'stays_in_weekend_nights']
]

,hotel,is_canceled,adr,adults,children,babies,stays_in_week_nights,stays_in_weekend_nights
48515,City Hotel,1,5400.00,2,0.0,0,1,0
111403,City Hotel,0,510.00,1,0.0,0,1,0
15083,Resort Hotel,0,508.00,2,0.0,0,1,0
103912,City Hotel,0,451.50,2,2.0,0,1,1
13142,Resort Hotel,1,450.00,2,0.0,0,10,4
13391,Resort Hotel,1,437.00,2,2.0,0,4,2
39155,Resort Hotel,0,426.25,2,2.0,0,6,2
39568,Resort Hotel,0,402.00,3,1.0,0,3,2
39118,Resort Hotel,0,397.38,3,2.0,0,5,3
13323,Resort Hotel,1,392.00,2,1.0,0,8,2


Пока **не удаляем** `adr == 0`, отрицательный `adr` или очень высокий `adr`. Сначала фиксируем их как подозрительные значения, а окончательное решение примем после EDA/анализа контекста.


## 5. Бронирования с нулевым количеством гостей

Пересчитываем напрямую по исходным числовым признакам, а не через производный `has_children`.


In [14]:
zero_guests = df[
    (df['adults'] == 0) &
    (df['children'].fillna(0) == 0) &
    (df['babies'] == 0)
]

print("Количество строк с 0 гостей:", len(zero_guests))
print("Доля:", len(zero_guests) / len(df))

zero_guests.head()

Количество строк с 0 гостей: 180
Доля: 0.001507663958455482


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,company,customer_type,adr,required_car_parking_spaces,total_of_special_requests,has_children,has_agent,has_company,has_agent_or_company,country_group
2224,Resort Hotel,0,1,2015,October,41,6,0,3,0,...,174,Transient-Party,0.0,0,0,0,0,1,1,Domestic
2409,Resort Hotel,0,0,2015,October,42,12,0,0,0,...,174,Transient,0.0,0,0,0,0,1,1,Domestic
3181,Resort Hotel,0,36,2015,November,47,20,1,2,0,...,<NA>,Transient-Party,0.0,0,0,0,1,0,1,Foreign
3684,Resort Hotel,0,165,2015,December,53,30,1,4,0,...,<NA>,Transient-Party,0.0,0,0,0,1,0,1,Domestic
3708,Resort Hotel,0,165,2015,December,53,30,2,4,0,...,<NA>,Transient-Party,0.0,0,0,0,1,0,1,Domestic


Посмотрим, есть ли у этих строк закономерности.


In [15]:
print("Target:")
print(zero_guests['is_canceled'].value_counts(normalize=True, dropna=False))

print("\nHotel:")
print(zero_guests['hotel'].value_counts(dropna=False))

print("\nMarket segment:")
print(zero_guests['market_segment'].value_counts(dropna=False))

print("\nADR:")
print(zero_guests['adr'].describe())

Target:
is_canceled
0    0.861111
1    0.138889
Name: proportion, dtype: float64

Hotel:
hotel
City Hotel      167
Resort Hotel     13
Name: count, dtype: int64

Market segment:
market_segment
Online TA        69
Offline TA/TO    37
Direct           24
Groups           20
Complementary    15
Corporate        13
Aviation          2
Name: count, dtype: int64

ADR:
count    180.000000
mean      10.456500
std       31.681635
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max      200.000000
Name: adr, dtype: float64


Пока эти строки не удаляем. Они выглядят подозрительно, но без дополнительного подтверждения мы не можем доказать, что это техническая ошибка.


## 6. Дубликаты

Дубликаты считаем на **исходном `raw_df`**, потому что после удаления признаков разные строки могут искусственно стать одинаковыми.


In [16]:
raw_duplicate_count = raw_df.duplicated().sum()
raw_duplicate_share = raw_duplicate_count / len(raw_df)

print("Полных совпадений в raw_df:", raw_duplicate_count)
print("Доля:", raw_duplicate_share)

Полных совпадений в raw_df: 31994
Доля: 0.26797889270458164


Посмотрим несколько групп полных совпадений.


In [17]:
raw_duplicates = raw_df[raw_df.duplicated(keep=False)].sort_values(
    list(raw_df.columns)
)

raw_duplicates.head(20)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
40772,City Hotel,0,0,2015,August,32,7,0,2,2,...,No Deposit,14.0,NaN,0,Transient,75.0,0,1,Check-Out,2015-08-09
40802,City Hotel,0,0,2015,August,32,7,0,2,2,...,No Deposit,14.0,NaN,0,Transient,75.0,0,1,Check-Out,2015-08-09
40821,City Hotel,0,0,2015,August,32,8,0,1,2,...,No Deposit,9.0,NaN,0,Transient,89.0,0,1,Check-Out,2015-08-09
40838,City Hotel,0,0,2015,August,32,8,0,1,2,...,No Deposit,9.0,NaN,0,Transient,89.0,0,1,Check-Out,2015-08-09
76792,City Hotel,0,0,2015,August,33,10,1,0,2,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-08-11
76793,City Hotel,0,0,2015,August,33,10,1,0,2,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-08-11
76794,City Hotel,0,0,2015,August,33,10,1,0,2,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-08-11
41067,City Hotel,0,0,2015,August,33,11,0,1,1,...,No Deposit,NaN,38.0,0,Transient-Party,88.0,0,0,Check-Out,2015-08-12
41073,City Hotel,0,0,2015,August,33,11,0,1,1,...,No Deposit,NaN,38.0,0,Transient-Party,88.0,0,0,Check-Out,2015-08-12
41071,City Hotel,0,0,2015,August,33,11,0,1,2,...,No Deposit,NaN,NaN,0,Transient,80.0,0,0,Check-Out,2015-08-12


**Пока не удаляем дубликаты.** В датасете нет `booking_id`, поэтому полное совпадение признаков не доказывает, что это одна и та же реальная бронь. Это могут быть разные бронирования с одинаковыми записанными характеристиками.


## 7. Проверка оставшихся пропусков


In [18]:
missing_after = pd.DataFrame({
    'count': df.isna().sum(),
    'percent': df.isna().mean() * 100
})

missing_after[missing_after['count'] > 0].sort_values('percent', ascending=False)

,count,percent
company,112593,94.306893
agent,16340,13.686238
children,4,0.003350


## 8. Итог Data Cleaning Investigation

### Явный leakage
- `reservation_status`
- `reservation_status_date`
- `booking_changes`
- `assigned_room_type`
- `days_in_waiting_list`

Они удалены из рабочего `df`, потому что недоступны сразу после создания бронирования или сообщают итог.

### Преобразования
- создал `has_children`;
- создал `has_agent`, `has_company`, `has_agent_or_company`;
- `agent` и `company` перевёл в категориальный тип;
- пропуски `country` выделил в отдельную категорию `Unknown`;
- создал дополнительный признак `is_domestic`.

### Выглядит подозрительно
- экстремальные значения `adr`;
- очень большие значения `adults`, `babies`, `lead_time`, количества ночей;
- бронирования с нулём гостей;
- `Undefined` в категориальных признаках;
- полные совпадения строк в исходных данных.

### Что решил не удалять и почему
- `children`: количество детей содержит больше информации, чем только бинарный `has_children`;
- `agent` и `company`: ID могут содержать полезный сигнал, несмотря на пропуски;
- `country`: высокая кардинальность сама по себе не является причиной для удаления;
- дубликаты: без `booking_id` невозможно доказать, что одинаковые строки — одна и та же реальная бронь;
- `adr == 0` и бронирования с 0 ночей: сначала нужно исследовать контекст, а не считать их ошибкой автоматически.


## 9. Сохраняем датасет


In [19]:
Path("../data/interim").mkdir(parents=True, exist_ok=True)

df.to_parquet(
    "../data/interim/hotel_bookings_clean.parquet",
    index=False
)
